In [9]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
from config import PATH_RAW_DATA, CSV_ENCODING, CSV_SEPARATOR, NA_MARKERS

pd.set_option('display.max_columns', None)

# 1.Extract

In [10]:
df = pd.read_csv(PATH_RAW_DATA, encoding=CSV_ENCODING, sep=CSV_SEPARATOR, na_values=NA_MARKERS)

# 2.Transform

- Trata coluna "nome_cientifico" pois alguns valores possuem "." ao final
- Substituir vírgula por pontos
- Utilizar a coluna "nome_cientifico" como "genero" utilizando split()
- Alterar linha 264(id_especie=202) especie deve ser "Pouteria"
- Alterar dtype das colunas numericas

In [11]:
# Colunas de identificação (não entram no cálculo de similaridade, só como chave/rótulo)
id_cols = [
    'id_especie',
    'nome_cientifico',
    'nome_popular_1',
    'genero',
    'especie',
    'familia'
]

# Núcleo físico-mecânico (baixa taxa de missing, base da similaridade)
numeric_features = [
    'densidade_basica',
    'contracao_tangencial',
    'contracao_radial',
    'contracao_volumetrica',
    'relacao_tangencial_radial',
    'flexao_seca_moe',
    'flexao_seca_mor',
    'compressao_paralela_seca',
    'dureza_janka_paralela_seca',
    'dureza_janka_transversal_seca',
    'cisalhamento_seca'
]

# Estético/sensorial (categóricas, relevantes se o uso-alvo for móveis/acabamento)
categorical_features = [
    'cor_cerne_classificacao',
    'textura',
    'gra',
    'brilho',
    'cerne_alburno',
    'figura_tangencial'
]

# Opcional: viabilidade de processamento (mais missing, usar com cautela/imputação)
processing_features = [
    'secagem_duracao_dias'
]

# Array combinado para o dataset de trabalho
colunas_selecionadas = id_cols + numeric_features + categorical_features + processing_features

df = df[colunas_selecionadas]
df.head(1)

,id_especie,nome_cientifico,nome_popular_1,genero,especie,familia,densidade_basica,contracao_tangencial,contracao_radial,contracao_volumetrica,relacao_tangencial_radial,flexao_seca_moe,flexao_seca_mor,compressao_paralela_seca,dureza_janka_paralela_seca,dureza_janka_transversal_seca,cisalhamento_seca,cor_cerne_classificacao,textura,gra,brilho,cerne_alburno,figura_tangencial,secagem_duracao_dias
0,21,Astronium lecointei,MUIRACATIARA-RAJADA,Astronium,lecointei,Anacardiaceae,"0,75","7,2","4,1",11,"1,76","12,94","145,63","84,14","7688,43","8659,29","11,77",marrom,media,"ondulada, sem desenho",alto nas superfícies longitudinais,NaN,NaN,NaN


## 2.1 - Função para tratar coluna "nome_cientifico"

In [ ]:
'''
Remove o "." ao final de alguns valores para padronização
'''
def normalize_col(df:pd.DataFrame) -> pd.DataFrame:
    mask = df["nome_cientifico"].str.endswith(".", na=False)
    
    if mask.any():
        df.loc[mask, "nome_cientifico"] = df.loc[mask, "nome_cientifico"].str.rstrip(".")
    else:
        print(df.loc[:, "nome_cientifico"])
    return df


## 2.2 - Função para consertar coluna "gênero"

In [ ]:
'''
Na linha do id_especie = 202:
    nome_cientifico = "Pouteria guianensis" ✅ (correto)
    genero = "Guianensis " ❌ (deveria ser "Pouteria")
    especie = "" ❌ (vazio; deveria ser "guianensis")
'''
df[df["nome_cientifico"]=="Pouteria guianensis"]

,id_especie,nome_cientifico,nome_popular_1,genero,especie,familia,densidade_basica,contracao_tangencial,contracao_radial,contracao_volumetrica,relacao_tangencial_radial,flexao_seca_moe,flexao_seca_mor,compressao_paralela_seca,dureza_janka_paralela_seca,dureza_janka_transversal_seca,cisalhamento_seca,cor_cerne_classificacao,textura,gra,brilho,cerne_alburno,figura_tangencial,secagem_duracao_dias
264,202,Pouteria guianensis,ABIURANA,Guianensis,NaN,Sapotaceae,0.64,"8,95","5,09","13,34","1,76",NaN,"140,53","68,06","11797,43","12993,84","18,93",NaN,média,ondulada,moderado,distintos,"pouco destacada, causada pelas linhas vasculares",NaN


## 2.9 - Chamada das funções

In [55]:
normalize_col(df)

0                                    Astronium lecointei
1                                     Bagassa guianensis
2                                       Bowdichia nitida
3                                Calophyllum brasiliense
4                                      Carapa guianensis
                             ...                        
264                                  Pouteria guianensis
265                                Pouteria oblanceolata
266                              Qualea brevipedicellata
267    Tachigali chrysophylla =Sclerolobium chrysophy...
268         Tachigali glauca = Tachigali cf myrmecophila
Name: nome_cientifico, Length: 269, dtype: str


,id_especie,nome_cientifico,nome_popular_1,genero,especie,familia,densidade_basica,contracao_tangencial,contracao_radial,contracao_volumetrica,relacao_tangencial_radial,flexao_seca_moe,flexao_seca_mor,compressao_paralela_seca,dureza_janka_paralela_seca,dureza_janka_transversal_seca,cisalhamento_seca,cor_cerne_classificacao,textura,gra,brilho,cerne_alburno,figura_tangencial,secagem_duracao_dias
0,21,Astronium lecointei,MUIRACATIARA-RAJADA,Astronium,lecointei,Anacardiaceae,0.64,"7,2","4,1",11,"1,76","12,94","145,63","84,14","7688,43","8659,29","11,77",marrom,media,"ondulada, sem desenho",alto nas superfícies longitudinais,NaN,NaN,NaN
1,24,Bagassa guianensis,TATAJUBA,Bagassa,guianensis,Moraceae,0.64,"5,8","4,1","9,5","1,41","11,57","124,45","78,55","9875,32","7384,43","12,55",amarela,"média, com desenho em estrias nas superfícies ...",entrecruzada geralmente,alto nas superfícies longitudinais,distintos,NaN,NaN
2,28,Bowdichia nitida,SUCUPIRA,Bowdichia,nitida,Fabaceae,0.64,"7,4","4,5","12,3","1,64","13,53","153,96","86,79","12758,48","11307,1","12,55",marrom,grossa com pouco desenho,direita a ligeiramente entrecruzada,alto,distintos,NaN,NaN
3,47,Calophyllum brasiliense,JACAREÚBA,Calophyllum,brasiliense,Calophyllaceae,0.64,"8,4","5,4","12,9","1,56","8,53","87,67","53,25","7864,95","5668,26","10,59",marrom,média e homogênea,"irregular e, geralmente, entrecruzada",moderado,pouco distintos a indistintos,NaN,NaN
4,50,Carapa guianensis,ANDIROBA,Carapa,guianensis,Meliaceae,0.64,7,"4,6","11,8","1,52","10,3","94,83","53,45","8080,7","6295,89","9,61",marrom,média,direita a entrecruzada,fraco,indistintos,pouco desenho,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,202,Pouteria guianensis,ABIURANA,Guianensis,NaN,Sapotaceae,0.64,"8,95","5,09","13,34","1,76",NaN,"140,53","68,06","11797,43","12993,84","18,93",NaN,média,ondulada,moderado,distintos,"pouco destacada, causada pelas linhas vasculares",NaN
265,203,Pouteria oblanceolata,TUTURUBÁ,Pouteria,oblanceolata,Sapotaceae,0.64,"8,9","5,2","13,8","1,71","15,4","153,18","77,86","13199,78","13003,65","15,98",NaN,fina,direita,ausente,pouco distintos,"pouco destacada, em linhas longitudinais ondul...",8
266,215,Qualea brevipedicellata,MANDIOQUEIRA,Qualea,brevipedicellata,Vochysiaceae,0.64,"8,6","4,4","13,6","1,95","15,59",132,"79,83","11032,51","9630,15","13,14",NaN,média a grossa,cruzada reversa,moderado,distintos,"de aspecto fibroso acentuado, causado por dest...",32
267,226,Tachigali chrysophylla =Sclerolobium chrysophy...,TAXI-PITOMBA,Tachigali,chrysophylla,Caesalpiniaceae,0.64,"7,4","3,7",11,2,"12,26","115,72","59,13","7668,82","5952,65","13,44",marrom,média,cruzada irregular,forte,pouco distintos,linhas (radial),"11,5"


In [ ]:
#%reset -f